In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator

hbar_c = 197.3269804  # MeV*fm

#------------------------------------------------

#Create DataFrame
df = pd.read_csv("EOS.dat", sep=r"\s+", header=None, names=["n_b", "en", "P"])

#Remove very small baryon density values n_b
df = df[df["n_b"] > 1e-6].copy()       #Create an independent copy of the filtered DataFrame

#Compute chemical potential
df["xm"] = (df["en"] + df["P"]) / df["n_b"]

#Sort rows by chemical potential (ascending order)
df = df.sort_values("xm")
df = df.drop_duplicates(subset="xm", keep="first")  #Avoid duplicate μ values

#Convert pandas columns to NumPy arrays
xm_np = df["xm"].to_numpy()
P_np  = df["P"].to_numpy()

#Construct hadronic pressure function using PCHIP interpolation
P_h = PchipInterpolator(xm_np, P_np) 

#------------------------------------------------

#Quark model 
def P_q(a4, D, Beff_14, xm):

    """Parameters:
       xm        : MeV            ,χημικό δυναμικό
       a4        : dimensionless  
       D         : MeV
       Beff_14  : MeV             ,(B_eff^(1/4)) """
   
    Beff = Beff_14**4
    P_q_g = 3/(4*(np.pi**2)) * a4 * ((xm/3)**4) + 3/((np.pi**2)) * (D**2) * (xm/3)**2 - Beff
    P_q = P_q_g /(hbar_c**3)         #MeV/fm3
    return P_q

#------------------------------------------------

#Root finding using the bisection method
def bisection_root(f, a, b, tol=1e-6, max_iter=200):
    
    #Initial bracket check
    if f(a) * f(b) > 0:
        return None
        
    #Iterative procedure
    for _ in range(max_iter):
        m = (a + b)/2         
        
        #Convergence criterion
        if abs(f(m)) < tol or (b - a) < tol:
            return m
            
        #Update interval
        if f(a) * f(m) < 0:
            b = m
        else:
            a = m
            
    return (a + b)/2           #Return midpoint if maximum iterations reached 

#Compute first crossover 
def find_first_intersection_bisection(a4, D, Beff_14, xm_grid, P_min=1e-3):

    #Define pressure difference between quark and hadronic matter
    def f(xm):
        return P_q(a4, D, Beff_14, xm) - P_h(xm)              # >0: quark, <0: hadronic 
        
    #Compute difference over chemical potential grid (xm_grid)
    f_xm_grid = f(xm_grid)
    
    for i in range(len(f_xm_grid) - 1):
        if f_xm_grid[i] == 0:
            a = xm_grid[i]
            b = xm_grid[i]
            break
        elif f_xm_grid[i] * f_xm_grid[i+1] < 0:
            a = xm_grid[i]
            b = xm_grid[i+1]
        else:
            continue

        if a == b:
            xm_star = a
        else:
            xm_star = bisection_root(f, a, b, tol=1e-6, max_iter=200) #Bisection root search in [a, b]
            if xm_star is None:
                continue

        #Compute pressure at intersection point
        P_star = float(P_h(xm_star))

        #Avoid trivial root at P=0
        if P_star <= P_min:
            continue
        
        return xm_star, P_star

    return None

#------------------------------------------------

#Define the chemical potential range
xm_min = max(900, float(xm_np.min()))
xm_max = min(1600, float(xm_np.max()))
xm_grid = np.linspace(xm_min, xm_max, 3000)


#Define parameters (CFL)
parameter_sets = [
    (0.7,   0, 180),
    (0.7,  25, 170),
    (0.7,  50, 160),
    (0.7,  50, 170),
    (0.7,  50, 180),
    (0.7,  75, 170),
    (0.3, 150, 180)]

#------------------------------------------------

#Collect transition pressure (Ptr) values
Ptr_list = []
xmtr_list = []

#Plot
fig1 = plt.figure(figsize=(8, 5))
plt.plot(xm_np, P_np, "k--", linewidth=2, label="APR (EOS.dat)")   #

for a4, D, Beff_14 in parameter_sets:
    line, = plt.plot(xm_grid,P_q(a4, D, Beff_14, xm_grid),label=f"CFL ({a4}, {D}, {Beff_14})")
    color = line.get_color()

    #Compute Crossover
    crossover = find_first_intersection_bisection(a4, D, Beff_14, xm_grid, P_min=1e-3)
    if crossover is not None:
        xm_star, P_star = crossover
        xmtr_list.append(xm_star)
        Ptr_list.append(P_star)
        plt.scatter([xm_star], [P_star],color=color,s=20)
        print(f"CFL ({a4}, {D}, {Beff_14}) -> xm*={xm_star:.2f} MeV, P*={P_star:.2f} MeV/fm^3")
    else:
        print(f"CFL ({a4}, {D}, {Beff_14}) -> no intersection in [{xm_min:.1f}, {xm_max:.1f}] MeV")

plt.xlabel(r"$\mu$ [MeV]")
plt.ylabel(r"$P$ [MeV/fm$^3$]")
plt.title(r"P–$\mu$ with APR–CFL intersections")
plt.xlim(900, 1600)
plt.ylim(-50, 350)
plt.grid(True)
plt.legend(fontsize=8)
plt.tight_layout()
fig1.savefig('P - μ with APR–CFL intersections.pdf', dpi=80, bbox_inches='tight')
plt.show()

print("Ptr", Ptr_list)
print("Xm", xmtr_list)